<a href="https://www.kaggle.com/code/asgx1224/hftbacktest-claude?scriptVersionId=356768926" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# 1. 安装 hftbacktest。Kaggle 每次重开会话都要重新运行这一行。
#    -q 表示安静模式，只显示错误，不输出安装过程的大量日志。
!pip install -q hftbacktest

# 2. 确认安装成功，打印版本号。
#    importlib.metadata 是 Python 标准库，用来读取已安装包的版本。
from importlib.metadata import version
print("hftbacktest 版本:", version("hftbacktest"))

# 3. 打开 npz 文件。
#    .npz 本质是多个 numpy 数组打包在一起的压缩文件，
#    np.load 只是打开它，此时还没有把全部数据读进内存。
import numpy as np
path = "/kaggle/input/datasets/asgx1224/hftbacktest/BTCUSDT_2023-04-01.npz"
f = np.load(path)

# 4. 列出文件里有哪些数组（键名）。
print("包含的键:", f.files)

# 5. 逐个查看每个数组的形状（行数、列数）和数据类型。
#    f[k] 这时才会真正读取该数组，一次只读一个，避免内存爆掉。
for k in f.files:
    arr = f[k]
    print(k, "形状:", arr.shape, "类型:", arr.dtype)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 61.5 MB/s eta 0:00:00
hftbacktest 版本: 2.4.4
包含的键: ['data']
data 形状: (60943533,) 类型: [('ev', '<u8'), ('exch_ts', '<i8'), ('local_ts', '<i8'), ('px', '<f8'), ('qty', '<f8'), ('order_id', '<u8'), ('ival', '<i8'), ('fval', '<f8')]


In [2]:
import pandas as pd

data = f["data"]   # 取出数组。这一步不复制数据，只是拿到引用。

# 直接打印前 5 行，每一行是一个事件，显示全部字段。
print(data[:5])

# 把交易所时间戳（纳秒整数）转换成日期时间，便于阅读。
# unit="ns" 告诉 pandas 输入的整数单位是纳秒。
print(pd.to_datetime(data["exch_ts"][:5], unit="ns"))

# 只看 ev 字段的原始值，十六进制显示，方便之后对照位标志。
print([hex(x) for x in data["ev"][:5]])

[(3489660930, 1680307200158000000, 1680307200165963000, 28454.8, 0.009, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200173448000, 28454.8, 0.027, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200173512000, 28454.8, 0.005, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200174315000, 28454.8, 0.022, 0, 0, 0.)
 (3489660930, 1680307200183000000, 1680307200190901000, 28454.8, 0.004, 0, 0, 0.)]
DatetimeIndex(['2023-04-01 00:00:00.158000', '2023-04-01 00:00:00.165000',
               '2023-04-01 00:00:00.165000', '2023-04-01 00:00:00.165000',
               '2023-04-01 00:00:00.183000'],
              dtype='datetime64[ns]', freq=None)
['0xd0000002', '0xd0000002', '0xd0000002', '0xd0000002', '0xd0000002']


In [3]:
import numpy as np                      # 数值库，下面用 np.round 取整
from numba import njit                  # numba 的即时编译装饰器，让循环跑得像编译语言一样快
from hftbacktest import BacktestAsset, HashMapMarketDepthBacktest

@njit                                   # 告诉 numba：把下面这个函数编译成机器码
def print_bbo(hbt, n):                  # hbt 是回测对象，n 是最多打印几次
    count = 0                           # 计数器，记录已经打印了几次
    while hbt.elapse(60 * 1e9) == 0:    # 时间前进 60 秒（单位纳秒）；返回 0 表示还没读完数据，继续循环
        depth = hbt.depth(0)            # 取第 0 个资产（我们只有一个）当前的盘口
        print(
            hbt.current_timestamp,              # 回测当前时间，纳秒时间戳
            np.round(depth.best_bid, 1),        # 最优买价，保留 1 位小数（价格是浮点数，可能有微小误差）
            np.round(depth.best_ask, 1),        # 最优卖价，同样保留 1 位小数
        )
        count += 1                      # 计数加 1
        if count >= n:                  # 打印够 n 次就跳出循环
            break
    return True                         # numba 函数建议有明确的返回值

asset = (
    BacktestAsset()
    .data([path])                       # 数据文件路径，字符串即可，库会按需分段读取
    .linear_asset(1.0)                  # 线性合约，合约面值 1.0
    .tick_size(0.1)                     # 最小价格变动
    .lot_size(0.001)                    # 最小下单量
)

hbt = HashMapMarketDepthBacktest([asset])   # 创建回测对象；参数是资产列表
print_bbo(hbt, 5)                           # 只打印前 5 次，也就是前 5 分钟
hbt.close()                                 # 回测对象不能重复使用，用完必须关闭

1680307260158000000 28474.7 28474.8
1680307320158000000 28455.1 28455.2
1680307380158000000 28453.9 28454.0
1680307440158000000 28444.3 28444.4
1680307500158000000 28442.4 28442.5


0

In [4]:
from hftbacktest import FILLED,GTC, LIMIT, NONE, NEW

@njit
def rest_at_best_bid(hbt, max_seconds):       # max_seconds：最多观察多少秒，防止永远等不到成交
    order_id = 1
    submitted = False
    last_status = -1                          # 上一次看到的订单状态
    seconds = 0                               # 提交订单之后已经过去的秒数
    while hbt.elapse(1e9) == 0:               # 每次前进 1 秒，没读完数据就继续
        depth = hbt.depth(0)
        if not np.isfinite(depth.best_bid):   # 盘口还没建好就跳过（上一轮讲过）
            continue
        if not submitted:
            price = depth.best_bid            # 买价就设成当前最优买价，也就是排在队尾
            hbt.submit_buy_order(0, order_id, price, 0.01, GTC, LIMIT, False)
            submitted = True
            print("提交", hbt.current_timestamp, "价格", np.round(price, 1))
        else:
            order = hbt.orders(0).get(order_id)          # 取出我们的订单
            if order.status != last_status:              # 状态变了才打印
                last_status = order.status               # 更新“上一次状态”
                print(hbt.current_timestamp, "状态", order.status,
                      "最优买", np.round(depth.best_bid, 1),
                      "最优卖", np.round(depth.best_ask, 1),
                      "持仓", hbt.position(0))           # position(0)：第 0 个资产的持仓，单位 BTC
            if order.status == FILLED:                   # 成交了就结束
                return True
            seconds += 1                                 # 计时加 1 秒
            if seconds >= max_seconds:                   # 超时还没成交，也结束
                return False
    return False

hbt = HashMapMarketDepthBacktest([asset])     # asset 沿用之前的定义；hbt 已关闭，要重新创建
print("是否成交:", rest_at_best_bid(hbt, 1800))   # 最多观察 1800 秒，也就是 30 分钟
hbt.close()

提交 1680307201158000000 价格 28454.8
1680307202158000000 状态 1 最优买 28454.8 最优卖 28454.9 持仓 0.0
1680307320158000000 状态 3 最优买 28455.1 最优卖 28455.2 持仓 0.01
是否成交: True


0

In [5]:
ev = data["ev"]                                      # 事件类型位标志数组
is_trade = (ev & 0xFF) == 2                          # 低 8 位等于 2：成交事件
is_exch  = (ev & 0x80000000) != 0                    # 带 EXCH 标志：每笔成交只会有一行带它，避免重复计数
is_sell  = (ev & 0x10000000) != 0                    # 带 SELL 标志：卖方主动成交
in_win   = (data["exch_ts"] >= 1680307319000000000) & (data["exch_ts"] <= 1680307320200000000)  # 119 秒到 120.2 秒
low_px   = data["px"] <= 28454.85                    # 价格不高于我们的 28454.8（留一点浮点误差余量）

sel = data[is_trade & is_exch & is_sell & in_win & low_px]   # & 把几个条件同时满足的行筛出来
print("符合条件的成交行数:", len(sel))
for r in sel[:30]:                                   # 最多打印前 30 行
    print(r["exch_ts"], r["px"], r["qty"])

符合条件的成交行数: 19
1680307320094000000 28454.8 0.015
1680307320094000000 28454.8 0.045
1680307320094000000 28454.8 0.015
1680307320094000000 28454.8 0.002
1680307320094000000 28454.8 0.015
1680307320094000000 28454.8 0.015
1680307320094000000 28454.8 0.031
1680307320094000000 28454.8 0.014
1680307320094000000 28454.8 0.015
1680307320094000000 28454.8 0.032
1680307320094000000 28454.8 0.014
1680307320094000000 28454.8 0.001
1680307320094000000 28454.8 0.082
1680307320094000000 28454.7 0.008
1680307320094000000 28454.7 0.006
1680307320094000000 28454.7 0.047
1680307320094000000 28454.7 0.419
1680307320097000000 28454.7 0.005
1680307320109000000 28454.7 0.001
